# Lab 5 — Finding the right passage: building and measuring retrieval

**Artificial Intelligence · 4th year · UIR · Pr. Hakim Hafidi**

| | |
|---|---|
| **Duration** | 2 hours, individual work |
| **Runtime** | CPU is enough — no GPU, no API key, no quota today |
| **You need** | the four course study sheets (`Session_1.pdf` … `Session_4.pdf`) |
| **You will learn** | how to turn documents into searchable passages · how to measure whether a search works · why one embedding model is not interchangeable with another · how to combine two kinds of search |
| **Deliverable** | this notebook, executed, + `lab05_<your id>.json`, both on connect.uir.ac.ma |
| **Grading** | 4 points: it runs (1) · measurements are correct (1) · interpretation (1) · critical analysis (1) |

> **No generation today.** Not one word will be produced by a language model. That is
> deliberate: next session you will put retrieved passages into a prompt, and whoever arrives
> with a weak retriever will build a weak assistant without understanding why.
>
> **You write three functions today**, each from a written contract, each checked by a set of
> tests you must make pass. That is the work — not filling in single lines.

---
## Step 0 — Setup (10 minutes)

Run the install cell, then upload the four study-sheet PDFs. Select all four at once in the
file picker.

In [1]:
%pip install -q -U sentence-transformers rank-bm25 pypdf
print("done — if Colab asks you to restart the session, do it, then re-run this cell.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 16.2 MB/s eta 0:00:00
done — if Colab asks you to restart the session, do it, then re-run this cell.


In [ ]:
import pathlib, os
from google.colab import files

existing = sorted(pathlib.Path(".").glob("Session_*.pdf"))
if len(existing) < 4:
    print("Select Session_1.pdf, Session_2.pdf, Session_3.pdf and Session_4.pdf (all four at once):")
    files.upload()

PDFS = sorted(pathlib.Path(".").glob("Session_*.pdf"))
assert len(PDFS) == 4, f"Found {len(PDFS)} PDFs, expected 4: {[p.name for p in PDFS]}"
print("✅", [p.name for p in PDFS])

In [ ]:
#@title ⚙️ Lab 5 toolbox — run this cell once, then fold it { display-mode: "form" }
import json, re, time, pathlib, unicodedata
import numpy as np, pandas as pd

CALL_LOG = []   # no external API in this lab

# ---------------------------------------------------------------- reading PDFs
def extract_pdf(path):
    """Return the text of a PDF, page by page, joined with newlines.

    One normalisation is done for you: PDF extraction frequently returns words separated by
    tabs or runs of spaces (an artefact of justified text), which would silently break every
    string match you write later. Horizontal whitespace is collapsed to single spaces; line
    breaks are kept, because you need them in Part A. Everything else is left exactly as the
    extractor produced it — this is the unglamorous step where most real RAG systems already
    go wrong."""
    from pypdf import PdfReader
    reader = PdfReader(str(path))
    pages = []
    for p in reader.pages:
        t = p.extract_text() or ""
        t = "\n".join(re.sub(r"[ \t\xa0]+", " ", line).strip() for line in t.split("\n"))
        pages.append(t)
    return "\n".join(pages)

# ---------------------------------------------------------------- embeddings
_models = {}
def get_model(name):
    """Load a sentence-transformer once and keep it. Runs on CPU; first load downloads ~90 MB."""
    from sentence_transformers import SentenceTransformer
    if name not in _models:
        print(f"loading {name} …")
        _models[name] = SentenceTransformer(name)
    return _models[name]

def embed(texts, model_name="sentence-transformers/all-MiniLM-L6-v2", batch_size=32):
    """Turn a list of strings into a matrix of unit-length vectors (one row per string).
    Rows are normalised, so a dot product between two rows IS their cosine similarity."""
    m = get_model(model_name)
    v = m.encode(list(texts), batch_size=batch_size, show_progress_bar=False,
                 convert_to_numpy=True, normalize_embeddings=True)
    return v.astype(np.float32)

def cosine_search(query_vec, chunk_matrix, top_k=10):
    """Return (indices, scores) of the top_k most similar chunks, best first.
    Both arguments must come from embed(), so vectors are already normalised."""
    scores = chunk_matrix @ query_vec
    idx = np.argsort(-scores)[:top_k]
    return idx.tolist(), scores[idx].tolist()

# ---------------------------------------------------------------- keyword search
_bm25_cache = {}
def bm25_index(chunks):
    """Build a classic keyword index over the chunks (BM25, 1994 — still hard to beat on names)."""
    from rank_bm25 import BM25Okapi
    key = id(chunks)
    if key not in _bm25_cache:
        _bm25_cache[key] = BM25Okapi([tokenize(c) for c in chunks])
    return _bm25_cache[key]

def tokenize(text):
    """Lowercase, strip accents, keep words and numbers. Used by BM25 only."""
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(c for c in text if not unicodedata.combining(c))
    return re.findall(r"[a-z0-9]+", text)

def bm25_search(query, chunks, top_k=10):
    """Return (indices, scores) of the top_k chunks by keyword match, best first."""
    scores = np.array(bm25_index(chunks).get_scores(tokenize(query)))
    idx = np.argsort(-scores)[:top_k]
    return idx.tolist(), scores[idx].tolist()

# ---------------------------------------------------------------- the evaluation set
def find_gold(chunks, snippet):
    """Indices of the chunks that contain `snippet` (case-insensitive).

    The evaluation set marks a correct answer by a short piece of text, NOT by a chunk
    number — because chunk numbers change every time you re-chunk. This is what lets you
    sweep chunk sizes in Part D and still compare the results."""
    s = snippet.lower().strip()
    return [i for i, c in enumerate(chunks) if s in c.lower()]

def show_chunk(chunks, i, width=300):
    c = re.sub(r"\s+", " ", chunks[i])
    return f"[{i}] {c[:width]}{'…' if len(c) > width else ''}"

def export_lab(lab_id, student_id, answers: dict, tables: dict = None, extra: dict = None):
    student_id = str(student_id).strip()
    assert student_id and student_id.lower() not in ("", "xxxxx"), "Set STUDENT_ID."
    payload = {"lab": lab_id, "student_id": student_id,
               "answers": {k: str(v).strip() for k, v in answers.items()},
               "tables": {k: (v.to_dict("records") if hasattr(v, "to_dict") else v)
                          for k, v in (tables or {}).items()},
               "extra": extra or {}, "call_log": CALL_LOG,
               "created": time.strftime("%Y-%m-%d %H:%M")}
    name = f"{lab_id}_{student_id}.json"
    pathlib.Path(name).write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    short = [k for k, v in payload["answers"].items() if len(v) < 40]
    print(f"📦 {name} written ({pathlib.Path(name).stat().st_size} bytes)")
    print("⚠️  too short:", short) if short else print("✅ All answers filled.")
    print(f"\nSubmit the notebook AND {name} on connect.uir.ac.ma")
    return payload

print("✅ Lab 5 toolbox loaded: extract_pdf, embed, cosine_search, bm25_search, find_gold")

In [ ]:
STUDENT_ID = "XXXXX"   # TODO: your apogee number
assert STUDENT_ID.strip().upper() != "XXXXX", "Set STUDENT_ID first."
print("ok")

---
## Part A — The corpus, and what extraction does to it (15 minutes)

Your corpus is the four study sheets of Sessions 1 to 4. You know their content, which is why
writing questions with known answers will take you five minutes instead of thirty.

Extract the text and **look at it** before doing anything else. Extraction is where most real
retrieval systems are already broken, and nobody notices because nobody reads the output.

In [ ]:
raw = {p.stem: extract_pdf(p) for p in PDFS}

for name, text in raw.items():
    print(f"{name}: {len(text):>6} characters, {len(text.split()):>5} words")
print(f"\nTOTAL: {sum(len(t.split()) for t in raw.values())} words\n")

print("First 600 characters of Session_1 as extracted:\n")
print(raw["Session_1"][:600])

### A.1 — Find the repeated text

Look carefully at the extraction above and at the output below. Something appears on **every
page** of every document, and it has nothing to do with the content.

In [ ]:
from collections import Counter
lines = [l.strip() for t in raw.values() for l in t.split("\n") if l.strip()]
repeated = [(n, l) for l, n in Counter(lines).most_common(8) if n > 2]
print("Lines appearing more than twice across the corpus:\n")
for n, l in repeated:
    print(f"  {n:>3}×  {l[:90]!r}")

In [ ]:
# TODO (3–6 lines): write `clean_text()`.
#
#   CONTRACT
#   --------
#   Input : the raw text of one document
#   Output: the same text with the running header and footer removed, and runs of
#           whitespace collapsed to single spaces where appropriate.
#   Remove at least: the page header, the page footer, and the "page N / M" marker.
#   Do NOT remove section headings — they are content.
#
#   Hint: re.sub with a pattern per piece of furniture is enough. Look at what A.1 printed.

def clean_text(text):
    ...

assert isinstance(clean_text("a test string"), str),     "clean_text has no body yet — replace the `...` with your implementation."

docs = {name: clean_text(t) for name, t in raw.items()}

# --- tests you must make pass -------------------------------------------------
for name, t in docs.items():
    assert "Pr. Hakim Hafidi" not in t, f"{name}: the header is still there"
    assert "study sheet" not in t.lower() or t.lower().count("study sheet") <= 1, \
        f"{name}: the footer is still there"
    assert not re.search(r"page\s+\d+\s*/\s*\d+", t), f"{name}: the page marker is still there"
    assert len(t.split()) > 2000, f"{name}: you removed too much ({len(t.split())} words left)"
    assert "The thread of this session" in t, f"{name}: you removed real content"
print("✅ clean_text passes all tests")
for name, t in docs.items():
    print(f"   {name}: {len(raw[name].split())} → {len(t.split())} words")

---
## Part B — Write the chunker (20 minutes)

A retriever does not search documents; it searches **passages**. How you cut the documents
decides what can ever be found. Too small and a passage loses its context; too big and the
relevant sentence drowns in noise and the prompt fills up.

You write the chunker. Read the contract, then make all seven tests pass.

In [ ]:
# TODO: write `chunk_text()`.
#
#   CONTRACT
#   --------
#   chunk_text(text, size, overlap) -> list[str]
#
#   · Cut `text` into pieces of at most `size` WORDS.
#   · Consecutive pieces share `overlap` words: the last `overlap` words of a piece are
#     also the first `overlap` words of the next one. This stops a sentence that straddles
#     a boundary from being lost by both pieces.
#   · Never return an empty or whitespace-only piece.
#   · Every word of `text` appears in at least one piece — nothing is dropped.
#   · If the text is shorter than `size`, return exactly one piece.
#   · `overlap` must be strictly smaller than `size`; raise ValueError otherwise.
#   · overlap=0 means no repetition between consecutive pieces.
#
#   Write it for a list of words, not for characters — it is much easier.

def chunk_text(text, size=200, overlap=40):
    ...

In [ ]:
# --- the seven tests ----------------------------------------------------------
assert isinstance(chunk_text("a b c", size=2, overlap=0), list),     "chunk_text has no body yet — replace the `...` with your implementation."

w = " ".join(f"w{i}" for i in range(1000))

c = chunk_text(w, size=100, overlap=0)
assert all(len(x.split()) <= 100 for x in c), "T1: a piece is longer than `size`"
assert len(c) == 10, f"T1: expected 10 pieces with size=100 overlap=0, got {len(c)}"

c = chunk_text(w, size=100, overlap=20)
assert len(c) > 10, "T2: with overlap you need MORE pieces to cover the same text"
first_tail = c[0].split()[-20:]
second_head = c[1].split()[:20]
assert first_tail == second_head, "T3: consecutive pieces must share exactly `overlap` words"

assert all(x.strip() for x in chunk_text(w, 100, 20)), "T4: empty piece returned"

covered = set()
for x in chunk_text(w, 100, 20): covered.update(x.split())
assert covered == set(w.split()), "T5: some words are in no piece at all"

assert len(chunk_text("three words only", size=100, overlap=10)) == 1, \
    "T6: a text shorter than `size` must give exactly one piece"

try:
    chunk_text(w, size=50, overlap=50); raise AssertionError("T7: overlap >= size must raise")
except ValueError:
    pass

print("✅ chunk_text passes all seven tests")

In [ ]:
def build_chunks(docs, size, overlap):
    """Chunk every document and remember which document each chunk came from."""
    chunks, source = [], []
    for name, text in docs.items():
        for c in chunk_text(text, size, overlap):
            chunks.append(c); source.append(name)
    return chunks, source

chunks, source = build_chunks(docs, size=200, overlap=40)
print(f"{len(chunks)} chunks from {len(docs)} documents")
print(f"average length: {np.mean([len(c.split()) for c in chunks]):.0f} words\n")
print(show_chunk(chunks, 5))

---
## Part C — Build an evaluation set, and write the metric (20 minutes)

You cannot improve what you cannot measure, and you cannot measure retrieval without knowing
the right answer in advance. So: ten questions whose answers you know, and a metric.

**How a question is marked.** Not by a chunk number — those change every time you re-chunk —
but by a short **snippet** of text that the right passage must contain. `find_gold()` then
locates the matching chunks for any chunking. This is what makes Part D possible.

Five questions are given. **You write five more**, of which **two in French** — you will need
those in Part E.

In [ ]:
EVAL = [
    {"q": "Which report triggered the first AI winter?",
     "snippet": "commissioned the Lighthill", "doc": "Session_1"},
    {"q": "Why can a single perceptron not solve XOR?",
     "snippet": "one straight line", "doc": "Session_2"},
    {"q": "What is the difference between precision and recall?",
     "snippet": "of those flagged", "doc": "Session_3"},
    {"q": "How many parameters did GPT-3 have?",
     "snippet": "300 billion training tokens", "doc": "Session_4"},
    {"q": "What does a model do when the classes are imbalanced and you only look at accuracy?",
     "snippet": "always answers", "doc": "Session_3"},

    # TODO: add FIVE questions of your own, TWO of them written in French.
    #   · `q`       : the question, as a user would type it — do NOT copy the words of the
    #                 passage, or you are testing nothing.
    #   · `snippet` : a short, distinctive piece of text from the passage that answers it.
    #                 Keep it under ~6 words and make sure it appears only where you mean.
    #   · `doc`     : which study sheet it comes from.
    #
    # Example of a BAD question: "What is the knowledge-acquisition bottleneck?" with snippet
    # "knowledge-acquisition bottleneck" — the question repeats the answer's words, so even a
    # keyword search finds it trivially and you learn nothing.

]

assert len(EVAL) == 10, f"You need 10 questions, you have {len(EVAL)}"
fr = [e for e in EVAL if e.get("lang") == "fr" or re.search(r"\b(quel|quelle|pourquoi|comment|qu[\u2019\']est)\b", e["q"], re.I)]
assert len(fr) >= 2, "At least two questions must be written in French"

for e in EVAL:
    g = find_gold(chunks, e["snippet"])
    assert g, f"No chunk contains {e['snippet']!r} — check the spelling against the PDF"
    assert len(g) <= 4, f"{e['snippet']!r} appears in {len(g)} chunks — make it more specific"
print(f"✅ evaluation set valid: {len(EVAL)} questions, {len(fr)} in French")

In [ ]:
# TODO: write `recall_at_k()`.
#
#   CONTRACT
#   --------
#   recall_at_k(rankings, golds, k) -> float
#
#   · rankings : one list of chunk indices per question, best first (what the search returned)
#   · golds    : one list of correct chunk indices per question (what find_gold returned)
#   · returns  : the fraction of questions for which AT LEAST ONE correct chunk appears in
#                the first k results. A float between 0 and 1.
#   · If a ranking is shorter than k, use all of it.
#   · A question with no gold chunk at all must raise ValueError — a silent zero would hide
#     a broken evaluation set, which is worse than a crash.

def recall_at_k(rankings, golds, k):
    ...

# --- tests --------------------------------------------------------------------
assert isinstance(recall_at_k([[1]], [[1]], 1), float),     "recall_at_k has no body yet, or does not return a float."

assert recall_at_k([[1, 2, 3]], [[1]], k=1) == 1.0, "R1: gold in first position"
assert recall_at_k([[1, 2, 3]], [[3]], k=1) == 0.0, "R2: gold outside top 1"
assert recall_at_k([[1, 2, 3]], [[3]], k=3) == 1.0, "R3: gold inside top 3"
assert recall_at_k([[1, 2], [5, 6]], [[2], [9]], k=2) == 0.5, "R4: one of two questions found"
assert recall_at_k([[1]], [[1, 7]], k=5) == 1.0, "R5: k larger than the ranking is fine"
assert recall_at_k([[4, 1]], [[1, 4]], k=1) == 1.0, "R6: any one gold chunk is enough"
try:
    recall_at_k([[1, 2]], [[]], k=2); raise AssertionError("R7: empty gold must raise")
except ValueError:
    pass
print("✅ recall_at_k passes all tests")

In [ ]:
def evaluate_retrieval(chunks, eval_set, search_fn, k_values=(1, 3, 5, 10)):
    """Run every question through `search_fn` and report recall at several k."""
    rankings, golds = [], []
    for e in eval_set:
        idx, _ = search_fn(e["q"], chunks)
        rankings.append(idx)
        golds.append(find_gold(chunks, e["snippet"]))
    return {f"recall@{k}": round(recall_at_k(rankings, golds, k), 3) for k in k_values}, rankings, golds

MODEL_EN = "sentence-transformers/all-MiniLM-L6-v2"

def make_vector_search(model_name=MODEL_EN):
    """Return a search function that embeds the chunks once and reuses them."""
    cache = {}
    def search(query, chunks, top_k=10):
        key = (id(chunks), model_name)
        if key not in cache:
            cache[key] = embed(chunks, model_name)
        qv = embed([query], model_name)[0]
        return cosine_search(qv, cache[key], top_k)
    return search

vector_search = make_vector_search()
baseline, rankings, golds = evaluate_retrieval(chunks, EVAL, vector_search)
print("Baseline — vector search, chunks of 200 words with 40 of overlap:")
print(baseline)

---
## Part D — Does chunking matter? Measure it (20 minutes)

You chose 200 words and 40 of overlap because the notebook told you to. Now find out whether
that was a good choice, by sweeping the two parameters and measuring each configuration on
your own ten questions.

Nine configurations, re-chunked and re-embedded each time. It takes a couple of minutes.

In [ ]:
# TODO (5–8 lines): fill the sweep.
#   For each size in SIZES and each overlap in OVERLAPS:
#     · rebuild the chunks with build_chunks(docs, size, overlap)
#     · evaluate them with evaluate_retrieval(..., make_vector_search())
#     · append a row with: size, overlap, n_chunks, and the recall@k values
#   Note: overlap must stay smaller than size — skip the pairs that are not valid.

SIZES = [100, 200, 400]
OVERLAPS = [0, 40, 100]

rows = []
for size in SIZES:
    for ov in OVERLAPS:
        ...

sweep = pd.DataFrame(rows)
print(sweep.to_string(index=False))

In [ ]:
assert len(sweep) >= 7, (f"Expected at least 7 valid configurations, got {len(sweep)} — "
                        "did you fill in the loop body above?")
best = sweep.sort_values(["recall@3", "recall@5"], ascending=False).iloc[0]
print(f"\nBest on recall@3: size={int(best['size'])}, overlap={int(best['overlap'])} "
      f"→ recall@3 = {best['recall@3']}, {int(best['n_chunks'])} chunks")

# TODO (2 lines): choose the configuration you will use for the rest of the lab, and say why.
#   It does not have to be the top row — a configuration with slightly lower recall but far
#   fewer chunks may be the better engineering choice. Justify whichever you pick.
MY_SIZE, MY_OVERLAP = ..., ...
MY_CHOICE_REASON = "..."

assert MY_OVERLAP < MY_SIZE and len(MY_CHOICE_REASON) > 40, "Pick a valid pair and justify it."
chunks, source = build_chunks(docs, MY_SIZE, MY_OVERLAP)
print(f"\nUsing {len(chunks)} chunks of {MY_SIZE} words with {MY_OVERLAP} of overlap.")

---
## Part E — The embedding model is a decision, not a default (20 minutes)

`all-MiniLM-L6-v2` was trained almost entirely on English. Your corpus is in English — but
two of your questions are in French, as a Moroccan user would naturally write them.

Compare it with a multilingual model on the **same** chunks and the **same** questions, then
look separately at what happens to the French questions. You saw the same effect collapse a
classifier in Lab 4; here it is a retrieval problem.

In [ ]:
MODEL_MULTI = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# TODO (4–8 lines): compare the two embedding models.
#   For each of MODEL_EN and MODEL_MULTI:
#     · build a search function with make_vector_search(model_name)
#     · evaluate it on the whole EVAL set
#     · evaluate it again on the FRENCH questions only
#   Collect rows with: model, recall@3 (all), recall@3 (French only).
#   Hint: evaluate_retrieval accepts any list of questions as its second argument.

rows = []
for model_name in [MODEL_EN, MODEL_MULTI]:
    ...

embed_cmp = pd.DataFrame(rows)
print(embed_cmp.to_string(index=False))

In [ ]:
assert len(embed_cmp) == 2, "One row per model."
assert "french" in " ".join(embed_cmp.columns).lower(), "You must report the French-only score."
print("\n✅ Part E done. Note the gap on the French questions specifically.")

---
## Part F — Two kinds of search, and how to combine them (25 minutes)

Vector search understands meaning: it finds "how long before I can leave" for a question about
notice periods. It is weak on exact strings — a part number, a law article, a proper name the
embedding model never saw during training.

Keyword search (BM25, 1994) is the opposite: it matches tokens exactly and understands
nothing. On names and codes it is still hard to beat.

Serious systems run both and merge the results. The interesting part is **how** you merge
them.

### F.1 — Where each one wins

In [ ]:
def bm25_only(query, chunks, top_k=10):
    return bm25_search(query, chunks, top_k)

vec = make_vector_search()
res_vec, rank_vec, golds = evaluate_retrieval(chunks, EVAL, vec)
res_bm, rank_bm, _       = evaluate_retrieval(chunks, EVAL, bm25_only)
print(pd.DataFrame([{"method": "vector", **res_vec}, {"method": "BM25", **res_bm}]).to_string(index=False))

print("\nQuestion by question — rank of the first correct chunk (— means not in the top 10):")
for e, rv, rb, g in zip(EVAL, rank_vec, rank_bm, golds):
    pv = next((i + 1 for i, c in enumerate(rv) if c in g), None)
    pb = next((i + 1 for i, c in enumerate(rb) if c in g), None)
    print(f"  vec {str(pv or '—'):>3} | bm25 {str(pb or '—'):>3}   {e['q'][:62]}")

### F.2 — Why you cannot simply add the scores

A cosine similarity lives in [−1, 1]. A BM25 score lives in [0, ∞) and its scale depends on
the corpus, the query length and the word frequencies. Adding them means letting BM25 decide
everything whenever its numbers happen to be large.

The standard answer is to throw the scores away and keep only the **ranks**. Reciprocal rank
fusion scores each document by summing 1/(K + rank) over the lists it appears in, with K a
constant (60 by convention) that stops the first position from dominating.

You write it.

In [ ]:
# TODO: write `reciprocal_rank_fusion()`.
#
#   CONTRACT
#   --------
#   reciprocal_rank_fusion(rankings, K=60) -> list[int]
#
#   · rankings : a list of ranked lists of chunk indices, each best-first.
#                Example: [[4, 1, 7], [1, 9]]
#   · For every index, its score is the sum over the lists it appears in of 1 / (K + rank),
#     where rank is its 0-based position in that list.
#   · Returns every index that appears in at least one list, sorted by decreasing score.
#   · An index missing from a list simply contributes nothing for that list.
#   · Ties may be broken any way you like, but the function must be deterministic:
#     calling it twice with the same input returns the same list.

def reciprocal_rank_fusion(rankings, K=60):
    ...

# --- tests --------------------------------------------------------------------
assert isinstance(reciprocal_rank_fusion([[1]]), list),     "reciprocal_rank_fusion has no body yet — replace the `...` with your implementation."

out = reciprocal_rank_fusion([[1, 2, 3]])
assert out == [1, 2, 3], f"F1: a single list must come back unchanged, got {out}"

out = reciprocal_rank_fusion([[1, 2], [1, 3]])
assert out[0] == 1, f"F2: an item ranked first in both lists must win, got {out}"
assert set(out) == {1, 2, 3}, "F3: every item from every list must appear"

out = reciprocal_rank_fusion([[9, 1], [1, 9]])
assert set(out) == {1, 9}, "F4: symmetric input, both items present"

a = reciprocal_rank_fusion([[1, 2, 3], [3, 2, 1]])
b = reciprocal_rank_fusion([[1, 2, 3], [3, 2, 1]])
assert a == b, "F5: the function must be deterministic"

out = reciprocal_rank_fusion([[5], [5], [5], [7, 5]])
assert out[0] == 5, "F6: an item present in many lists must outrank one present in few"
print("✅ reciprocal_rank_fusion passes all tests")

In [ ]:
def hybrid_search(query, chunks, top_k=10):
    """Run both searches wide, then fuse their rankings."""
    iv, _ = vec(query, chunks, top_k=20)
    ib, _ = bm25_search(query, chunks, top_k=20)
    return reciprocal_rank_fusion([iv, ib])[:top_k], None

res_hy, rank_hy, _ = evaluate_retrieval(chunks, EVAL, hybrid_search)

methods = pd.DataFrame([{"method": "vector", **res_vec},
                        {"method": "BM25", **res_bm},
                        {"method": "hybrid (RRF)", **res_hy}])
print(methods.to_string(index=False))

assert len(methods) == 3, "Three methods compared."
print("\n✅ Part F done.")

---
## Part G — Where it still fails, and why (20 minutes)

An average hides everything interesting. Find the questions your best method handles worst,
read the chunks it returned, and diagnose the cause. This is the part that will save you time
in the project.

In [ ]:
best_rank = rank_hy if res_hy["recall@5"] >= res_vec["recall@5"] else rank_vec
best_name = "hybrid" if res_hy["recall@5"] >= res_vec["recall@5"] else "vector"

worst = []
for e, r, g in zip(EVAL, best_rank, golds):
    pos = next((i + 1 for i, c in enumerate(r) if c in g), 999)
    worst.append({"question": e["q"], "snippet": e["snippet"], "rank_of_first_gold": pos})
worst = pd.DataFrame(worst).sort_values("rank_of_first_gold", ascending=False)
print(f"Using the {best_name} ranking.\n")
print(worst.head(5).to_string(index=False))

print("\n" + "=" * 78)
print("The three hardest questions, with what the search actually returned:")
print("=" * 78)
for _, row in worst.head(3).iterrows():
    e = next(x for x in EVAL if x["q"] == row["question"])
    r = best_rank[EVAL.index(e)]
    g = find_gold(chunks, e["snippet"])
    print(f"\nQ: {e['q']}")
    print(f"   expected a chunk containing {e['snippet']!r} → chunk(s) {g}")
    for i in r[:3]:
        mark = "✅" if i in g else "  "
        print(f"   {mark} {show_chunk(chunks, i, 150)}")

In [ ]:
# TODO (3 lines): diagnose your three hardest questions.
#   For each, pick ONE cause and name it in a short sentence. The usual suspects:
#     · "split"      — the answer straddles two chunks, so neither contains it whole
#     · "vocabulary" — the question and the passage use different words for the same thing
#     · "ambiguous"  — several passages legitimately answer it; the gold snippet is arbitrary
#     · "multi-hop"  — answering needs two passages combined, which no single chunk holds
#     · "noise"      — the retrieved chunks are dominated by boilerplate or headings

DIAGNOSES = {
    "...": "...",      # question (or its first words) : cause + one sentence
}

assert len(DIAGNOSES) >= 3, "Diagnose at least three questions."
assert all(len(v) > 30 for v in DIAGNOSES.values()), "One real sentence per diagnosis."
for q, d in DIAGNOSES.items(): print(f"· {q[:60]}\n    {d}\n")

---
## Part H — Analysis (20 minutes)

In English, with your own numbers. Two or three sentences each, except Q5.

In [ ]:
# Q1 — What did the running header and footer do to your chunks before you removed them?
#      How many words did cleaning remove across the four documents, and why does that text
#      hurt retrieval specifically?
Q1 = """
...
"""

# Q2 — From your sweep: how much did recall@3 vary between the best and the worst
#      configuration? Quote both. What does overlap buy you, and what does it cost?
Q2 = """
...
"""

# Q3 — Compare the two embedding models on the whole set and on the French questions.
#      Quote the four numbers. What does that tell a team building an assistant for users
#      who write in French or Arabic?
Q3 = """
...
"""

# Q4 — Look at your question-by-question table in F.1. Name one question where BM25 beat the
#      vector search and one where the opposite happened, and explain each in terms of what
#      the two methods actually match on. Then say in one sentence why RRF merges ranks
#      rather than scores.
Q4 = """
...
"""

# Q5 — CRITICAL ANALYSIS. You will reuse this retriever next session, with a language model
#      on top. Write the paragraph you would give to a colleague inheriting it. It must state:
#      the configuration you chose and why, the recall you measured and ON WHAT (how many
#      questions, written by whom), at least two reasons your measurement is optimistic
#      compared with real use, and one thing you would test before trusting it.
Q5 = """
...
"""

In [ ]:
answers = {"Q1": Q1, "Q2": Q2, "Q3": Q3, "Q4": Q4, "Q5": Q5}
problems = [f"{k} too short" for k, v in answers.items() if len(v.strip()) < (250 if k == "Q5" else 120)]
problems += [f"{k} still has the placeholder" for k, v in answers.items() if "..." in v]
print("⚠️  Not ready:", problems) if problems else print("✅ Complete — run the export cell.")

---
## Submit

Run the cell below, then upload **both** files on connect.uir.ac.ma.

In [ ]:
export_lab("lab05", STUDENT_ID,
           answers={"Q1": Q1, "Q2": Q2, "Q3": Q3, "Q4": Q4, "Q5": Q5},
           tables={"sweep": sweep, "embedding_models": embed_cmp,
                   "methods": methods, "hardest": worst.head(5)},
           extra={"my_size": int(MY_SIZE), "my_overlap": int(MY_OVERLAP),
                  "choice_reason": MY_CHOICE_REASON,
                  "n_chunks": len(chunks),
                  "eval_set": EVAL,
                  "words_before": {k: len(v.split()) for k, v in raw.items()},
                  "words_after": {k: len(v.split()) for k, v in docs.items()},
                  "diagnoses": DIAGNOSES})

---
## Going further (optional, not graded)

- **Chunk on structure instead of word count.** Split on the `##` headings of the study sheets
  rather than every N words. Does recall improve? Why would it, and when would it not?
- **Add a third ranker.** Search the section titles only, fuse three rankings instead of two.
- **Tune K in RRF.** Sweep it from 1 to 200. What does a very small K do, and why is 60 the
  usual default?
- **Measure precision, not only recall.** Of the top 5 chunks returned, how many are actually
  relevant? Recall says you found it; precision says how much noise came with it — and next
  session, that noise goes into the prompt and gets paid for by the token.
- **Break your own evaluation set.** Rewrite three of your questions using words taken
  directly from the passage. Watch BM25 jump and conclude something about benchmark design.